# Symbiote v3: matched dense students with span criticality

Upload the matching `symbiote_v3_kaggle.zip` as a **private** Kaggle dataset and attach it.
Enable Internet and GPU T4 x2 (uses one GPU). Add and enable the
`JARVISLABS_API_KEY` secret. The key is never put in code or the dataset.

Use **Save Version > Save & Run All** for background execution. Do not also run
the interactive copy: those are separate jobs/ledgers. The run trains two arms,
compares v2/LLMLingua-2, freezes a shared selector using development only, evaluates
the held-out test, measures runtime, and exports results. Reader allowance: INR500.
No dataset recollection. An accuracy improvement is an experimental question.

On failure, preserve the saved output and return `v3_status.json` plus
`symbiote_v3_results.zip`. These are written by the runner even if the final cell
never executes. The full output tree retains optimizer checkpoints for resume.


In [ ]:
# First run: leave this False. For recovery, attach the previous notebook's FULL
# saved output and set True. Keep the SAME input ZIP/config and API ledger.
RESUME_FROM_SAVED_OUTPUT = False

from pathlib import Path
import hashlib, json, zipfile

expected = "55026cbd483fd46f2fabac47a2badafa92fc89332ed95c4a97cabe6a6be24441"
source = None
for candidate in sorted(Path("/kaggle/input").rglob("V3_KAGGLE_MANIFEST.json")):
    if hashlib.sha256(candidate.read_bytes()).hexdigest() == expected:
        source = candidate.parent
        script = (source / "final_eval/v3/kaggle_bootstrap.py").read_bytes()
        break
if source is None:
    for candidate in sorted(Path("/kaggle/input").rglob("symbiote_v3_kaggle.zip")):
        with zipfile.ZipFile(candidate) as archive:
            if "V3_KAGGLE_MANIFEST.json" not in archive.namelist():
                continue
            raw = archive.read("V3_KAGGLE_MANIFEST.json")
            if hashlib.sha256(raw).hexdigest() == expected:
                source = candidate
                script = archive.read("final_eval/v3/kaggle_bootstrap.py")
                break
if source is None:
    raise FileNotFoundError("Add the matching symbiote_v3_kaggle.zip as a Kaggle input dataset.")
if hashlib.sha256(script).hexdigest() != "fc06b8d63a59b5a484e05f4839de33afc6d875a8a5193519e1150b06341dad9d":
    raise ValueError("Transfer bootstrap checksum mismatch. Use the matching notebook and ZIP.")
namespace = {"__name__": "verified_v3_bootstrap"}
exec(compile(script, "verified_v3_bootstrap", "exec"), namespace)
namespace["install"](source, expected)
previous_runs = [p for p in Path("/kaggle/input").rglob("V3_KAGGLE_MANIFEST.json")
                 if hashlib.sha256(p.read_bytes()).hexdigest() == expected
                 and (p.parent / "outputs/v3/student_01/resolved_config.json").is_file()]
if previous_runs and not RESUME_FROM_SAVED_OUTPUT:
    raise RuntimeError("A previous full run is attached. Set RESUME_FROM_SAVED_OUTPUT=True to retain its checkpoint/cache/ledger.")
if RESUME_FROM_SAVED_OUTPUT:
    namespace["restore_saved_output"]("/kaggle/input", expected)


In [ ]:
# No manual decisions between development and test. Default total reader cap: INR500.
# Leave RETRY_FAILED False on the first run. Change only after inspecting an API failure.
RETRY_FAILED = False

import os, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

try:
    key = UserSecretsClient().get_secret("JARVISLABS_API_KEY")
except Exception:
    raise RuntimeError("Add and ENABLE the JARVISLABS_API_KEY secret under Add-ons > Secrets.") from None
if not key or not key.strip():
    raise RuntimeError("JARVISLABS_API_KEY is empty.")
env = os.environ.copy()
env["HF_HUB_DISABLE_XET"] = "1"
env["JARVISLABS_API_KEY"] = key.strip()
del key
root = Path("/kaggle/working/symbiote_v3")
command = [sys.executable, "-u", "-m", "final_eval.v3.kaggle_runner", "run"]
if RETRY_FAILED:
    command.append("--retry-failed")
result = subprocess.run(command, cwd=root, env=env)
del env
if result.returncode:
    raise RuntimeError("Run stopped safely. Keep the notebook output; return v3_status.json and symbiote_v3_results.zip. Do not restart with an empty ledger.")


In [ ]:
from pathlib import Path
import json
from IPython.display import display, FileLink

working = Path("/kaggle/working")
status = json.loads((working / "v3_status.json").read_text())
print("EXECUTION COMPLETED:", status["success"])
summary = working / "symbiote_v3/outputs/v3/student_01/test/report/summary.json"
if summary.exists():
    report = json.loads(summary.read_text())
    print("Accuracy conclusion:", report["primary"].get("verdict", "inspect report"))
    print(json.dumps(report["primary"].get("result", {}), indent=2))
print("Download the results ZIP and v3_status.json; keep this saved output version.")
display(FileLink(str(working / "symbiote_v3_results.zip")))
display(FileLink(str(working / "v3_status.json")))
